# 01.04 Tiling 可视化交互实验

## 小节概述

VectorAdd 需要把长度为 <code>N</code> 的连续数组先均分给 <code>blockDim</code> 个逻辑 Block，再把每个 Block 均分为 <code>tileCount</code> 个真实 GM Tile。本节把这组整数关系变成可交互的区间图：修改参数后立即看到合法性、Tile 长度、搬运字节数和目标区间。

完成本节后，你应能够：

1. 推导 <code>blockLength=N/blockDim</code> 与 <code>tileLength=blockLength/tileCount</code>；
2. 计算任意 Block、Tile 对应的全局左闭右开区间；
3. 区分真实 GM Tile 数与队列缓冲槽位数；
4. 判断当前连续 <code>DataCopy</code> 是否满足 32 Byte 对齐。

开始前请完成 [01.03 VectorAdd 算子实验](01.03_vector_add_operator.ipynb)。


## 教程内容

### 1. 建立切分约束

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>条件</th><th style='text-align: left;'>原因</th><th style='text-align: left;'>不满足时</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>N % blockDim == 0</code></td><td style='text-align: left;'>本实验要求各逻辑 Block 等长</td><td style='text-align: left;'>存在未覆盖的尾部元素</td></tr>
    <tr><td style='text-align: left;'><code>blockLength % tileCount == 0</code></td><td style='text-align: left;'>本实验要求本核各真实 GM Tile 等长</td><td style='text-align: left;'>存在尾 Tile</td></tr>
    <tr><td style='text-align: left;'><code>tileLength × 4 % 32 == 0</code></td><td style='text-align: left;'>当前 <code>float32</code> 连续 <code>DataCopy</code> 长度需 32 Byte 对齐</td><td style='text-align: left;'>应调整切分，或在支持非对齐 Shape 的实现中改用 <code>DataCopyPad</code></td></tr>
  </tbody>
</table>

这里的 <code>tileCount</code> 表示每个 Block 真正覆盖多少段 GM 数据，因此也是 <code>Process</code> 的循环次数。队列缓冲槽位数只决定 Local Memory 中有几块可轮换存储，不能乘入或除入 <code>tileCount</code>。

![默认参数的一维数组 Block 与 Tile 切分](./images/tiling_map.svg)


### 2. 用纯函数计算区间

先用一个不依赖界面组件的函数表达切分规则。它按依赖顺序检查三道条件，合法时返回所有长度与目标 Tile 的全局区间。纯函数也作为交互界面的后端；即使当前 Notebook 没有 <code>ipywidgets</code>，公式与静态可视化仍可运行。


In [ ]:
def derive_tiling(total_length, block_dim, tile_count, block_index=0, tile_index=0, element_bytes=4):
    values = [total_length, block_dim, tile_count, element_bytes]
    if any(not isinstance(value, int) or value <= 0 for value in values):
        raise ValueError('长度、Block 数、Tile 数和元素字节数必须是正整数')
    if total_length % block_dim != 0:
        raise ValueError('N 必须能被 blockDim 整除')
    block_length = total_length // block_dim
    if block_length % tile_count != 0:
        raise ValueError('blockLength 必须能被 tileCount 整除')
    tile_length = block_length // tile_count
    tile_bytes = tile_length * element_bytes
    if tile_bytes % 32 != 0:
        raise ValueError('本实验的连续 DataCopy 长度必须按 32 Byte 对齐')
    if not 0 <= block_index < block_dim:
        raise ValueError('blockIndex 超出范围')
    if not 0 <= tile_index < tile_count:
        raise ValueError('tileIndex 超出范围')

    block_start = block_index * block_length
    tile_start = block_start + tile_index * tile_length
    return {
        'block_length': block_length,
        'tile_length': tile_length,
        'tile_bytes': tile_bytes,
        'block_range': (block_start, block_start + block_length),
        'tile_range': (tile_start, tile_start + tile_length),
    }

default_result = derive_tiling(16384, 8, 8, block_index=2, tile_index=3)
print(default_result)


<strong>检查点：</strong> 默认参数应得到 <code>block_length=2048</code>、<code>tile_length=256</code>、<code>tile_bytes=1024</code>，Block 2 为 <code>[4096, 6144)</code>，其中 Tile 3 为 <code>[4864, 5120)</code>。


### 3. 绘制 Block/Tile 区间

图中每个矩形代表一个真实 GM Tile，纵向分行代表不同逻辑 Block；选中的 Tile 使用橙色。横轴是全局元素下标，不是 Byte 地址。为了避免参数过大时图形无法阅读，可视化最多显示 16 个 Block、每个 Block 32 个 Tile，但公式本身不受该显示上限影响。


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def plot_tiling(total_length=16384, block_dim=8, tile_count=8, block_index=2, tile_index=3):
    result = derive_tiling(total_length, block_dim, tile_count, block_index, tile_index)
    if block_dim > 16 or tile_count > 32:
        raise ValueError('为保证图形可读，交互图限制 blockDim<=16、tileCount<=32')

    fig, ax = plt.subplots(figsize=(12, max(3.0, block_dim * 0.55)))
    for block in range(block_dim):
        block_start = block * result['block_length']
        for tile in range(tile_count):
            start = block_start + tile * result['tile_length']
            selected = block == block_index and tile == tile_index
            rectangle = Rectangle(
                (start, block), result['tile_length'], 0.72,
                facecolor='#ff9f43' if selected else '#8ecae6',
                edgecolor='#1f2937', linewidth=0.55,
            )
            ax.add_patch(rectangle)
    ax.set_xlim(0, total_length)
    ax.set_ylim(block_dim, -0.25)
    ax.set_yticks([index + 0.36 for index in range(block_dim)])
    ax.set_yticklabels([f'Block {index}' for index in range(block_dim)])
    ax.set_xlabel('global element index')
    ax.set_title(
        f"blockLength={result['block_length']}, tileLength={result['tile_length']}, "
        f"tileBytes={result['tile_bytes']}; selected Tile={result['tile_range']}"
    )
    ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()
    print('Block range:', result['block_range'])
    print('Tile range :', result['tile_range'])
    return result

plot_tiling()


### 4. 交互调整参数

<code>ipywidgets</code> 是 Jupyter 的交互控件库。拖动滑块时，它会把当前值传给 <code>plot_tiling</code> 并重绘结果。若当前环境未安装该可选库，代码会给出清晰提示；上一节纯函数和静态图仍可用于完成实验。


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    total_widget = widgets.Dropdown(options=[8192, 16384, 32768, 65536], value=16384, description='N')
    block_widget = widgets.Dropdown(options=[1, 2, 4, 8, 16], value=8, description='blockDim')
    tile_widget = widgets.Dropdown(options=[1, 2, 4, 8, 16, 32], value=8, description='tileCount')
    block_index_widget = widgets.IntSlider(value=2, min=0, max=7, description='blockIndex')
    tile_index_widget = widgets.IntSlider(value=3, min=0, max=7, description='tileIndex')
    output = widgets.Output()

    def redraw(*_):
        block_index_widget.max = block_widget.value - 1
        tile_index_widget.max = tile_widget.value - 1
        with output:
            output.clear_output(wait=True)
            try:
                plot_tiling(
                    total_widget.value, block_widget.value, tile_widget.value,
                    block_index_widget.value, tile_index_widget.value,
                )
            except ValueError as error:
                print('INVALID:', error)

    for control in [total_widget, block_widget, tile_widget, block_index_widget, tile_index_widget]:
        control.observe(redraw, names='value')
    controls = widgets.HBox([
        widgets.VBox([total_widget, block_widget, tile_widget]),
        widgets.VBox([block_index_widget, tile_index_widget]),
    ])
    display(controls, output)
    redraw()
except ImportError:
    print('当前内核没有 ipywidgets；请直接修改 plot_tiling(...) 参数并重新运行上一单元。')


### 5. 观察三类非法参数

下面依次触发分核不整除、核内分 Tile 不整除和单次搬运未对齐。检查顺序很重要：每组参数应报告第一条无法继续推导的约束，而不是向下取整后画出一个遗漏数据的图。


In [ ]:
invalid_cases = [
    ('分核不完整', (16385, 8, 8)),
    ('核内分 Tile 不完整', (16384, 8, 7)),
    ('连续 DataCopy 未对齐', (8200, 8, 1)),
]
for name, arguments in invalid_cases:
    try:
        derive_tiling(*arguments)
        print(name, ': unexpected PASS')
    except ValueError as error:
        print(name, ':', error)


## 课后实践

<strong>独立输入：</strong> <code>N=28672</code>、<code>blockDim=4</code>、<code>tileCount=7</code>、<code>blockIndex=3</code>、<code>tileIndex=6</code>、<code>float32</code>。

<strong>任务：</strong>

1. 手算 <code>blockLength</code>、<code>tileLength</code>、<code>tileBytes</code>；
2. 手算 Block 3 与其中 Tile 6 的全局区间；
3. 调用 <code>plot_tiling</code> 核对结果，解释为什么右端点恰好等于 <code>N</code>；
4. 补全下面的 <code>tiling_record.py</code>，让它输出与可视化相同的结果。待填写文件必须通过 <code>%%writefile</code> 生成。

练习文件保存在按当前系统用户标识隔离的临时课程根目录中（支持 UID 的系统使用 UID，否则使用登录用户名），实际路径会由下一单元打印。


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

previous_repo = globals().get('REPO_ROOT')
try:
    notebook_start = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    notebook_start = cached_repo.resolve()
search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([notebook_start, *notebook_start.parents])
REPO_ROOT = next(
    (p for p in search_roots if (p / 'contrib/tutorials/data_structures_compute').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库中打开本 Notebook')
os.chdir(REPO_ROOT)
CHAPTER = REPO_ROOT / 'contrib/tutorials/data_structures_compute/01_basic_operations'

USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
PRACTICE = USER_TEMP_ROOT / '01_tiling_visualization_practice'
PRACTICE.mkdir(parents=True, exist_ok=True)
PRACTICE_FILE = PRACTICE / 'tiling_record.py'
print('practice file:', PRACTICE_FILE)


In [ ]:
%%writefile {PRACTICE_FILE}
N = 28672
BLOCK_DIM = 4
TILE_COUNT = 7
BLOCK_INDEX = 3
TILE_INDEX = 6
ELEMENT_BYTES = 4

# TODO 1：依次计算 block_length、tile_length 与 tile_bytes。
block_length = 0
tile_length = 0
tile_bytes = 0

# TODO 2：计算目标 Block 和 Tile 的左闭右开区间。
block_range = (0, 0)
tile_range = (0, 0)

print('block_length=', block_length)
print('tile_length=', tile_length)
print('tile_bytes=', tile_bytes)
print('block_range=', block_range)
print('tile_range=', tile_range)


In [ ]:
result = subprocess.run(
    [sys.executable, str(PRACTICE_FILE)],
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
expected = ['block_length= 7168', 'tile_length= 1024', 'tile_bytes= 4096',
            'block_range= (21504, 28672)', 'tile_range= (27648, 28672)']
print('PRACTICE PASS' if all(item in result.stdout for item in expected) else 'PRACTICE TODO')
plot_tiling(28672, 4, 7, 3, 6)


#### 独立完成后查看参考答案

将开关改为 <code>True</code> 后，代码单元会使用 <code>cat</code> 展示推导说明和参考程序。


In [ ]:
SHOW_ANSWER = False
if SHOW_ANSWER:
    subprocess.run(['cat', str(CHAPTER / 'answer/01.04_tiling_visualization/answers.md')], check=True)
    subprocess.run(['cat', str(CHAPTER / 'answer/01.04_tiling_visualization/tiling_record.py')], check=True)
else:
    print('参考答案保持隐藏；独立完成后将 SHOW_ANSWER 改为 True。')


## 本节小结

本节把 <code>N → Block → Tile</code> 的公式变成了可交互区间图，并把整除、索引范围和 32 Byte 搬运对齐放入同一套可执行检查。图中 Tile 的数量始终表示真实 GM 数据分段，不表示队列槽位。


完成后继续进入 [01.05 章节实践](01.05_chapter_test.ipynb)。
